# SurrealEngine: Hybrid Schema Industry Examples

This notebook demonstrates the versatility of `SurrealEngine` in handling diverse data modeling requirements, ranging from strict `SCHEMAFULL` relational models to flexible `SCHEMALESS` time-series and graph-based architectures.

We will port the full suite of industry-standard schemas from the SurrealDB documentation to show how `SurrealEngine` maps to these advanced patterns.

### Live Validation
Unlike previous examples, this notebook performs **live validation**. We connect to a memory-based SurrealDB instance and use `create_table()` to ensure the generated SurrealQL is valid and accepted by the database engine.

In [ ]:
from datetime import datetime
from typing import Optional, List, Any
from surrealengine import (
    Document, RelationDocument, 
    StringField, FloatField, IntField, BooleanField, 
    DateTimeField, ListField, DictField, ReferenceField, 
    ComputedField, RecordIDField, OptionField, PointField,
    Expr, Event, create_connection, generate_schema_statements
)

# 1. Setup a memory-based SurrealDB connection for live validation
conn = create_connection(
    url       = "ws://db:8000/rpc",
    namespace = "test_ns",
    database  = "test_db",
    username  = "root",
    password  = "secret",
    make_default = True,
)
await conn.connect()

async def print_schema(doc_cls, schemafull=True):
    print(f"--- Schema for {doc_cls.__name__} ({'SCHEMAFULL' if schemafull else 'SCHEMALESS'}) ---")
    
    # Verify validity by executing on the live memory DB
    try:
        await doc_cls.create_table(schemafull=schemafull)
        print("✅ Valid SurrealQL generated and accepted by SurrealDB.")
    except Exception as e:
        print(f"❌ Validation error: {e}")
    
    # Print the statements for inspection
    statements = generate_schema_statements(doc_cls, schemafull=schemafull)
    for stmt in statements:
        print(stmt)
    print()

## 1. Energy: Project Planning (Activities & Milestones)

This example showcases:
- `SCHEMAFULL` tables for strict project management.
- Complex `ComputedField` for duration and progress tracking.
- Graph relationships via `RelationDocument`.

In [ ]:
class Activity(Document):
    """An individual project activity or task."""
    name = StringField(required=True)
    description = OptionField(StringField())
    start = DateTimeField(required=True)
    end = DateTimeField(required=True)
    
    # Computed duration
    duration = ComputedField("end - start")
    
    # Progress assertion (0.0 to 1.0)
    progress = FloatField(default=0.0, assertion="$value >= 0.0 AND $value <= 1.0")
    
    # References to other tables
    assigned_to = OptionField(ReferenceField("Employee"))

class Project(Document):
    """A large-scale energy project."""
    name = StringField(required=True)
    
    # Computed counts from incoming relationships
    total_risks = ComputedField("count(<~risk_of)")
    open_incidents = ComputedField("count(<~incident_at[WHERE closed == false])")

class Milestone(Document):
    """A project milestone tracking multiple activities."""
    project = ReferenceField(Project, required=True)
    activities = ListField(ReferenceField(Activity), default=[])
    name = StringField(required=True)
    
    # VALUE example: Auto-update timestamp
    last_updated = DateTimeField(default=Expr.raw("time::now()"))
    
    # Computed progress across multiple related objects
    progress = ComputedField("math::mean(activities.progress)")
    is_complete = ComputedField("progress == 1.0")

class DependsOn(RelationDocument):
    """Dependency graph between activities."""
    class Meta:
        collection = "depends_on"
        in_table = Activity
        out_table = Activity

await print_schema(Activity)
await print_schema(Milestone)
await print_schema(DependsOn)

## 2. Energy: SCADA (Time-Series & Sensors)

This example showcases:
- `SCHEMALESS` (Hybrid) approach for flexible telemetry data.
- Table Events (Triggers) for alerting.
- Geospatial `PointField`.

In [ ]:
class Sensor(Document):
    """An industrial sensor located in the field."""
    name = StringField(required=True)
    location = PointField() # Longitude/Latitude
    tags = ListField(StringField(), default=[])

class Reading(Document):
    """High-frequency telemetry data."""
    class Meta:
        # Hybrid approach: Schemafull=False allows extra fields outside the definition
        strict = False
        events = [
            Event(
                name="high_pressure_alert",
                when="$event == 'CREATE' AND pressure > 100",
                then="CREATE alert CONTENT { sensor: $after.sensor, message: 'High pressure detected!', val: $after.pressure }"
            )
        ]

    sensor = ReferenceField(Sensor, required=True, define_schema=True)
    pressure = FloatField(define_schema=True) # Critical field tracked even in schemaless
    
    # Flexible payload for diverse sensor types
    telemetry = DictField(define_schema=True)

class Alert(Document):
    sensor = ReferenceField(Sensor)
    message = StringField()
    val = FloatField()

await print_schema(Sensor)
await print_schema(Reading, schemafull=False)
await print_schema(Alert)

## 3. Energy: Risk Management

This example showcases:
- Cross-table computations.
- Integer bounds validation via `assertion`.
- Back-reference counts.

In [ ]:
class Risk(Document):
    """Project risk assessment."""
    assessment = StringField(required=True)
    impact = IntField(required=True, assertion="$value INSIDE 1..5")
    probability = IntField(required=True, assertion="$value INSIDE 1..5")
    
    # Computed risk score
    score = ComputedField("impact * probability")
    
    # Project reference
    project = ReferenceField(Project, required=True)

class RiskOf(RelationDocument):
    class Meta:
        collection = "risk_of"
        in_table = Risk
        out_table = Project

await print_schema(Risk)
await print_schema(Project) # Notice 'total_risks' computed field

## 4. Supply Chain: Contracts & Deliverables

This example showcases:
- Complex subqueries in `ComputedField`.
- Multi-table relationship hierarchy.

In [ ]:
class Vendor(Document):
    name = StringField(required=True)
    rating = FloatField(default=5.0)

class Contract(Document):
    vendor = ReferenceField(Vendor, required=True)
    base_amount = FloatField(required=True)
    
    # Computed: Total amount including all change orders
    # Demonstrates subquery support in SurrealEngine computations
    total_amount = ComputedField(
        "base_amount + math::sum((SELECT VALUE amount FROM change_order WHERE contract = $parent.id))"
    )

class Deliverable(Document):
    contract = ReferenceField(Contract, required=True)
    name = StringField(required=True)
    due_date = DateTimeField(required=True)
    is_delivered = BooleanField(default=False)

class ChangeOrder(Document):
    """Modifications to an existing contract."""
    contract = ReferenceField(Contract, required=True)
    amount = FloatField(required=True)
    reason = StringField()

await print_schema(Contract)
await print_schema(ChangeOrder)

## 5. HSSE (Health, Safety, Security, & Environment)

This example showcases:
- Relation attributes (storing data on the edge).
- Linkage between employees and sites.

In [ ]:
class Employee(Document):
    name = StringField(required=True)
    role = StringField()

class IncidentAt(RelationDocument):
    """A safety incident involving an employee at a project site."""
    class Meta:
        collection = "incident_at"
        in_table = Employee
        out_table = Project
    
    severity = IntField(assertion="$value INSIDE 1..10")
    description = StringField()
    closed = BooleanField(default=False)
    reported_at = DateTimeField(default=Expr.raw("time::now()"))

await print_schema(IncidentAt)

## 6. Finance: Banking & Account Management

This example showcases:
- Multi-table relations (one-to-many poly-references).
- Hierarchy of currency types.

In [ ]:
class Bank(Document):
    name = StringField(required=True)
    swift_code = StringField(unique=True)

class Customer(Document):
    name = StringField(required=True)
    email = StringField(unique=True)

class Currency(Document):
    """Base class for various currencies."""
    code = StringField(required=True)
    symbol = StringField()
    class Meta:
        abstract = True

class JPY(Currency): code = StringField(default="JPY")
class USD(Currency): code = StringField(default="USD")
class EUR(Currency): code = StringField(default="EUR")

class Account(RelationDocument):
    """A customer's account in a specific currency."""
    class Meta:
        collection = "account"
        in_table = Customer
        # In SurrealDB, relations can point to multiple tables
        # SurrealEngine handles this via List of types in metadata
        out_table = [JPY, USD, EUR]
    
    balance = FloatField(default=0.0)
    last_transaction = DateTimeField(default=Expr.raw("time::now()"))

await print_schema(Account)

## 7. Gaming: RPG Quest & Inventory System

This example showcases:
- Polymorphic field contents.
- Complex graph of characters, items, and quests.

In [ ]:
class Character(Document):
    name = StringField(required=True)
    level = IntField(default=1)
    
    # Computed character stats
    strength = ComputedField("10 + (level * 2)")

class Item(Document):
    name = StringField(required=True)
    # Effect can be many different things: a dict of stats, or a simple status text
    effect = DictField()
    rarity = StringField(assertion="$value INSIDE ['COMMON', 'RARE', 'EPIC', 'LEGENDARY']")

class Quest(Document):
    title = StringField(required=True)
    reward_xp = IntField(default=100)

class Owns(RelationDocument):
    """Inventory relationship."""
    class Meta:
        collection = "owns"
        in_table = Character
        out_table = Item
    qty = IntField(default=1)

class QuestLog(RelationDocument):
    """Character quest progress."""
    class Meta:
        collection = "quest_log"
        in_table = Character
        out_table = Quest
    status = StringField(default="ACTIVE", assertion="$value INSIDE ['ACTIVE', 'COMPLETED', 'FAILED']")

await print_schema(Character)
await print_schema(Item)
await print_schema(QuestLog)

## 8. Aerospace: Telescope Tracking

This example showcases:
- Geospatial targeting.
- Time-series observations.

In [ ]:
class Telescope(Document):
    name = StringField(required=True)
    aperture = FloatField() # in meters
    current_target = PointField() # Celestial coordinates

class CelestialBody(Document):
    name = StringField(required=True)
    type = StringField() # STAR, PLANET, GALAXY

class Observation(RelationDocument):
    class Meta:
        collection = "observed"
        in_table = Telescope
        out_table = CelestialBody
    
    timestamp = DateTimeField(default=Expr.raw("time::now()"))
    coordinates = PointField() # Snapshot of where it was seen
    notes = StringField()

await print_schema(Telescope)
await print_schema(Observation)

## Summary of SurrealEngine Capabilities

Through these industry examples, we have seen that `SurrealEngine` effectively maps Python classes to complex SurrealDB features:
- **Schemafull/Schemaless**: Toggled via `strict` in Meta or `schemafull` in generator.
- **Assertions**: Complex logic via `assertion="..."` parameter on any field.
- **Computations**: Dynamic fields calculated at the database level via `ComputedField`.
- **Graph Relationships**: Deeply typed relationships via `RelationDocument` with support for multi-table edges.
- **Geospatial**: Native Support for `PointField`.
- **Events**: Table-level triggers registered in Model Meta.